# PROT-ON: structure-based detection of designer PROTein interface MutatiONs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CSB-KaracaLab/prot-on/blob/main/PROT-ON.ipynb)

PROT-ON takes the structure of a protein dimer, mutates every interface residue of one chain to the other 19 amino acids, and scores each mutant with [EvoEF1](https://github.com/tommyhuangthu/EvoEF) (bundled) or [FoldX](http://foldxsuite.crg.eu/) (your own licensed copy). It then reports the mutations that are predicted to enrich or deplete binding, filtered by their effect on the stability of the scanned chain.

**How to use this notebook**

1. Run the cells from top to bottom (in Colab: *Runtime → Run all*). With the default settings this runs the example complex.
2. To scan your own dimer, change the settings in step 2 and run steps 2 to 6 again.

Every run of step 3 is kept, so you can scan the same complex in several structures (a crystal structure and docking models, for instance) and compare the mutations they propose in step 5.

The example (532 mutations) takes about a minute on a laptop. Larger interfaces take proportionally longer. The notebook runs on Google Colab, Linux and macOS.

If you use PROT-ON, please cite: Koşaca M, Yılmazbilek İ, Karaca E (2023). PROT-ON: A structure-based detection of designer PROTein interface MutatiONs. *Frontiers in Molecular Biosciences* 10:1063971. [doi:10.3389/fmolb.2023.1063971](https://doi.org/10.3389/fmolb.2023.1063971)

## 1. Set up PROT-ON
Downloads PROT-ON (unless the notebook is already inside a copy of the repository), compiles EvoEF1 and installs the plotting packages.

In [ ]:
#@title Set up PROT-ON { display-mode: "form" }
import os, shutil, subprocess, sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/CSB-KaracaLab/prot-on.git"


def _find_repo():
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "proton.py").is_file() and (folder / "EvoEF1" / "src").is_dir():
            return folder
    return None


REPO = _find_repo()
if REPO is None:
    if not Path("prot-on/proton.py").is_file():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO_URL, "prot-on"], check=True)
    REPO = Path("prot-on").resolve()
os.chdir(REPO)
(REPO / "results").mkdir(exist_ok=True)

# EvoEF1 is shipped as source code and compiled here.
EVOEF = REPO / "EvoEF1" / "EvoEF"
if not EVOEF.is_file():
    if shutil.which("g++") is None:
        raise RuntimeError("A C++ compiler (g++) is needed to build EvoEF1, and none was found.")
    build = subprocess.run("g++ -O3 -ffast-math -w -o EvoEF src/*.cpp", shell=True,
                           cwd=REPO / "EvoEF1", capture_output=True, text=True)
    if build.returncode != 0 or not EVOEF.is_file():
        print(build.stderr[-3000:])
        raise RuntimeError("EvoEF1 could not be compiled.")


def _major(package):
    try:
        return int(version(package).split(".")[0])
    except PackageNotFoundError:
        return None


# PROT-ON saves its figures with plotly 5 and kaleido 0.2; newer kaleido needs a Chrome installation.
wanted = [name for name in ("numpy", "pandas") if _major(name) is None]
if _major("plotly") != 5 or _major("kaleido") != 0:
    wanted += ["plotly==5.24.1", "kaleido==0.2.1"]
if wanted:
    print("Installing:", ", ".join(wanted))
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *wanted], check=True)

RUNS = {}                                     # every scan of this session, for the comparison in step 5
print("PROT-ON is ready in", REPO)

## 2. Input and settings

**Structure.** PROT-ON needs a PDB file with exactly two chains. Keep `example-input/complex.pdb` to try the example. To scan your own dimer, tick `upload_pdb` (Colab), or put the path *or the web address* of a PDB file in `pdb_file`. A docking model works as well as an experimental structure: a HADDOCK model, for example, can be scanned straight from its result page by pasting the address of its PDB file.

A chain of the file can hold more than one polypeptide. An antibody whose heavy and light chains were both given the chain ID `A` is scanned as one molecule, so the whole paratope is covered in a single run.

**Settings.**
- `chain_ID`: the chain whose interface residues will be mutated.
- `cut_off`: two residues are at the interface if any of their atoms are within this distance (Å).
- `IQR`: multiplier of the interquartile range used to call the outliers (the enriching and depleting mutations) of the ΔΔG distribution.
- `algorithm`: `EvoEF1` works out of the box. `FoldX` needs your own academic-licensed Linux `foldx` executable and `rotabase.txt`; you will be asked to upload them. The FoldX route is experimental in this notebook.

**Optional PSSM filter.** To also filter the mutations by evolutionary information, provide the PSSM of the scanned chain as a comma-separated file (one row per residue of the chain, one column per amino acid), for example from [POSSUM](https://possum.erc.monash.edu/server.jsp). Tick `upload_pssm`, or type its path in `pssm_file`. For the example complex you can use `example-input/complex_chain_D_pssm.csv`.

**Following the HADDOCK antibody–antigen tutorial?** The cell below fetches the three structures used in its PROT-ON bonus task. Tick its box, run it, and it prints the paths to paste into `pdb_file`; the antibody is chain `A` in all three.

In [ ]:
#@title Optional: structures of the HADDOCK antibody–antigen tutorial { display-mode: "form" }
download_tutorial_structures = False #@param {type:"boolean"}
#@markdown The interleukin-1β / gevokizumab complex of the
#@markdown [HADDOCK2.4 antibody–antigen tutorial](https://www.bonvinlab.org/education/HADDOCK24/HADDOCK24-antibody-antigen-basic/#bonus-3-dissecting-the-interface-energetics-prot-on-analysis):
#@markdown the crystal structure and two docking models of it. Scan them one after another and compare them in step 5.
import io, urllib.request, zipfile

TUTORIAL = {
    "4G6M-matched.pdb": ("https://www.bonvinlab.org/education/HADDOCK24/HADDOCK24-antibody-antigen-basic/input-data.zip",
                         "HADDOCK24-antibody-antigen-basic/4G6M-matched.pdb", "crystal structure"),
    "cluster5_1.pdb": ("https://wenmr.science.uu.nl/haddock2.4/run/4242424242/184871-4G6M-Ab-Ag-NMR-Ab-no-passive/cluster5_1.pdb",
                       None, "best model of the top-ranked cluster"),
    "cluster2_1.pdb": ("https://wenmr.science.uu.nl/haddock2.4/run/4242424242/184871-4G6M-Ab-Ag-NMR-Ab-no-passive/cluster2_1.pdb",
                       None, "best model of the cluster with the lowest ligand-RMSD"),
}

if download_tutorial_structures:
    folder = REPO / "notebook_inputs" / "haddock_tutorial"
    folder.mkdir(parents=True, exist_ok=True)
    archives = {}
    for name, (url, member, description) in TUTORIAL.items():
        target = folder / name
        if not target.is_file():
            if member is None:
                with urllib.request.urlopen(url, timeout=120) as response:
                    target.write_bytes(response.read())
            else:
                if url not in archives:
                    with urllib.request.urlopen(url, timeout=120) as response:
                        archives[url] = zipfile.ZipFile(io.BytesIO(response.read()))
                target.write_bytes(archives[url].read(member))
        print(f"{target.relative_to(REPO).as_posix():52s} {description}")
    print("\nCopy one of these paths into pdb_file in the cell below, and set chain_ID to A (the antibody).")
else:
    print("Tick the box to download the structures of the HADDOCK antibody-antigen tutorial.")

In [ ]:
#@title Input and settings { display-mode: "form" }
pdb_file = "example-input/complex.pdb" #@param {type:"string"}
upload_pdb = False #@param {type:"boolean"}
chain_ID = "D" #@param {type:"string"}
cut_off = 5.0 #@param {type:"number"}
IQR = 1.5 #@param {type:"number"}
algorithm = "EvoEF1" #@param ["EvoEF1", "FoldX"]
pssm_file = "" #@param {type:"string"}
upload_pssm = False #@param {type:"boolean"}

import re
import urllib.request
import pandas as pd

os.chdir(REPO)
INPUTS = REPO / "notebook_inputs"
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")


def _upload(what):
    """Ask for one file in Colab and return (name, content)."""
    if not IN_COLAB:
        raise RuntimeError(f"Uploading works only in Google Colab. Type the path of {what} instead.")
    from google.colab import files
    print(f"Choose {what}:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(f"No file was uploaded for {what}.")
    name, content = next(iter(uploaded.items()))
    for leftover in uploaded:                 # Colab also writes the files into the working folder
        Path(leftover).unlink(missing_ok=True)
    return name, content


def _safe_stem(name):
    stem = re.sub(r"[^A-Za-z0-9_-]+", "_", Path(name).stem).strip("_")
    return stem or "complex"


# --- structure -------------------------------------------------------------
if upload_pdb:
    name, content = _upload("your PDB file")
    INPUTS.mkdir(exist_ok=True)
    PDB_PATH = INPUTS / (_safe_stem(name) + ".pdb")
    PDB_PATH.write_bytes(content)
elif re.match(r"https?://", pdb_file.strip()):
    url = pdb_file.strip()
    INPUTS.mkdir(exist_ok=True)
    PDB_PATH = INPUTS / (_safe_stem(url.split("?")[0].rstrip("/").split("/")[-1]) + ".pdb")
    print("Downloading", url)
    with urllib.request.urlopen(url, timeout=180) as response:
        PDB_PATH.write_bytes(response.read())
else:
    source = Path(pdb_file.strip()).expanduser()
    if not source.is_file():
        raise FileNotFoundError(f"PDB file not found: {pdb_file}")
    source = source.resolve()
    inside_repo = REPO in source.parents
    relative = source.relative_to(REPO).as_posix() if inside_repo else ""
    if inside_repo and re.fullmatch(r"[A-Za-z0-9_./-]+\.pdb", relative):
        PDB_PATH = source
    else:                                     # PROT-ON needs a simple file name ending in .pdb
        INPUTS.mkdir(exist_ok=True)
        PDB_PATH = INPUTS / (_safe_stem(source.name) + ".pdb")
        shutil.copyfile(source, PDB_PATH)
PDB_NAME = PDB_PATH.stem

residues = {}
with open(PDB_PATH) as handle:
    for line in handle:
        if line[:4] == "ATOM":
            residues.setdefault(line[21], set()).add(line[22:26])
chain_ID = chain_ID.strip()
chain_summary = ", ".join(f"{chain if chain.strip() else '(blank)'} ({len(found)} residues)"
                          for chain, found in residues.items())
if not residues:
    raise ValueError(f"{PDB_PATH.name} holds no ATOM records. PROT-ON needs a structure in the PDB format; "
                     "an mmCIF file has to be converted first.")
if len(residues) != 2:
    raise ValueError(f"PROT-ON works only with dimers, but {PDB_PATH.name} has {len(residues)} chains: "
                     f"{chain_summary}. Please keep only the two chains of interest in the file.")
if chain_ID not in residues:
    raise ValueError(f"Chain '{chain_ID}' is not in {PDB_PATH.name}. Its chains are: {chain_summary}.")
if cut_off <= 0 or IQR <= 0:
    raise ValueError("cut_off and IQR must be positive numbers.")

# --- optional PSSM ---------------------------------------------------------
PSSM_PATH = None
if upload_pssm or pssm_file.strip():
    if upload_pssm:
        INPUTS.mkdir(exist_ok=True)
        name, content = _upload("the PSSM file (.csv)")
        PSSM_PATH = INPUTS / "uploaded_pssm.csv"
        PSSM_PATH.write_bytes(content)
    else:
        PSSM_PATH = Path(pssm_file.strip()).expanduser().resolve()
        if not PSSM_PATH.is_file():
            raise FileNotFoundError(f"PSSM file not found: {pssm_file}")
    pssm = pd.read_csv(PSSM_PATH, sep=",")
    missing = [aa for aa in AMINO_ACIDS if aa not in pssm.columns]
    if missing:
        raise ValueError("The PSSM file must be comma separated with one column per amino acid. "
                         f"Missing columns: {', '.join(missing)}")
    if len(pssm) != len(residues[chain_ID]):
        raise ValueError(f"The PSSM has {len(pssm)} rows, but chain {chain_ID} has "
                         f"{len(residues[chain_ID])} residues. It needs one row per residue.")

# --- optional FoldX --------------------------------------------------------
if algorithm == "FoldX":
    for filename in ("foldx", "rotabase.txt"):
        target = REPO / "src" / filename
        if not target.is_file():
            if not IN_COLAB:
                raise FileNotFoundError(f"To use FoldX, place '{filename}' in {REPO / 'src'}")
            _, content = _upload(f"your FoldX file '{filename}'")
            target.write_bytes(content)
    (REPO / "src" / "foldx").chmod(0o755)

print(f"Structure : {PDB_PATH.name} with chains {chain_summary}")
print(f"Scanning  : chain {chain_ID} with {algorithm} (cut-off {cut_off} Å, IQR {IQR})")
print(f"PSSM      : {PSSM_PATH.name if PSSM_PATH else 'not used'}")

## 3. Run PROT-ON
Builds every interface mutant, scores it, and runs the statistical analysis. The line below the cell shows the progress.

In [ ]:
#@title Run PROT-ON { display-mode: "form" }
import signal, tempfile, threading, time

os.chdir(REPO)
results_root = REPO / "results"
prefix = f"{PDB_NAME}_chain_{chain_ID}"
output_name = f"{prefix}_{algorithm}_output"


def _work_folders():
    return {p for p in REPO.iterdir() if p.is_dir() and re.fullmatch(r"[0-9a-f]{32}", p.name)}


def _count_lines(path):
    with open(path) as handle:
        return sum(1 for line in handle if line.strip())


results_before = {p.name for p in results_root.iterdir()}
work_before = _work_folders()
root_copy = REPO / PDB_PATH.name              # proton.py leaves a copy of the input here
had_root_copy = root_copy.exists()
pssm_copy = None
if PSSM_PATH is not None:                     # the PSSM filter looks for this file name next to the scripts
    pssm_copy = REPO / "src" / f"{prefix}_pssm.csv"
    shutil.copyfile(PSSM_PATH, pssm_copy)

# PROT-ON starts its scripts with "python"; make sure that is this notebook's Python.
shim = Path(tempfile.mkdtemp(prefix="proton_python_"))
(shim / "python").write_text(f'#!/bin/sh\nexec "{sys.executable}" "$@"\n')
(shim / "python").chmod(0o755)
env = dict(os.environ, PYTHONUNBUFFERED="1", PATH=f"{shim}{os.pathsep}{os.environ.get('PATH', '')}")

command = [sys.executable, "proton.py", "--pdb", PDB_PATH.relative_to(REPO).as_posix(),
           "--chain_ID", chain_ID, "--algorithm", algorithm, "--cut_off", str(cut_off), "--IQR", str(IQR)]
log_path = REPO / "proton_run.log"
progress = {"stage": "Starting", "pass": 0, "built": 0, "contacts": 0}
STAGES = {"Defining the Interface": "Defining the interface",
          "Energies are being": "Computing binding and stability energies",
          "Outlier Detection Process": "Analysing the scores",
          "Box plot analysis": "Drawing the figures"}


def _follow(stream):
    with open(log_path, "w") as log:
        for line in stream:
            log.write(line)
            if "Distance is" in line:
                progress["contacts"] += 1
                continue
            built = re.match(r"EvoEF Building Mutation Model (\d+)", line)
            if built:
                number = int(built.group(1))
                if number == 1:
                    progress["pass"] += 1
                progress.update(stage="Building the mutant models", built=number)
                continue
            for marker, stage in STAGES.items():
                if line.startswith(marker):
                    progress["stage"] = stage


def _status(work, total, started):
    stage = progress["stage"]
    detail = ""
    scored = 0
    if work is not None:
        scored = len(list(work.glob("*_output/mutation_models/*.pdb"))) + len(list(work.glob("src/mutation_models/*.pdb")))
    of_total = f" of {total}" if total else ""
    if stage == "Building the mutant models":
        detail = f": {progress['built']}{of_total} ({'complex' if progress['pass'] < 2 else 'scanned chain alone'})"
    elif scored:
        if stage in ("Starting", "Defining the interface"):
            stage = "Building and scoring the mutants"
        detail = f": {scored}{of_total} mutations scored"
    minutes, seconds = divmod(int(time.time() - started), 60)
    return f"{stage}{detail} | {minutes:02d}:{seconds:02d} elapsed"


started = time.time()
process = subprocess.Popen(command, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, errors="replace", start_new_session=True)
reader = threading.Thread(target=_follow, args=(process.stdout,), daemon=True)
reader.start()
RESULT_DIR = None
try:
    work, total = None, None
    while process.poll() is None:
        time.sleep(1)
        if work is None:
            work = next(iter(_work_folders() - work_before), None)
        if work is not None and total is None:
            for candidate in (work / "src" / f"{prefix}_mutation_list", work / "EvoEF1" / f"{prefix}_mutation_list",
                              work / "EvoEF1" / "individual_list.txt"):
                if candidate.is_file():
                    total = _count_lines(candidate) or None
                    break
        print("\r" + _status(work, total, started).ljust(100), end="", flush=True)
    reader.join(timeout=10)
    print()
    new_results = [p for p in results_root.iterdir() if p.is_dir() and p.name not in results_before]
    finished = [p / output_name for p in new_results if (p / output_name / f"{prefix}_proton_scores").is_file()]
    if not finished:
        if progress["contacts"] == 0:
            print(f"No atoms of the two chains are within {cut_off} Å of each other, so there is no interface "
                  "to scan. Check the structure, or increase cut_off.\n")
        with open(log_path) as log:
            tail = [line.rstrip() for line in log if "Distance is" not in line][-25:]
        print("\n".join(tail))
        raise RuntimeError(f"PROT-ON did not finish. The last lines of its log are shown above; the full log is in {log_path}")
    RESULT_DIR = finished[0]
    shutil.move(str(log_path), str(RESULT_DIR / "proton_run.log"))
finally:
    if process.poll() is None:                # interrupted: stop PROT-ON and everything it started
        os.killpg(process.pid, signal.SIGTERM)
        process.wait()
    for folder in _work_folders() - work_before:
        shutil.rmtree(folder, ignore_errors=True)
    if pssm_copy is not None:
        pssm_copy.unlink(missing_ok=True)
    if not had_root_copy:
        root_copy.unlink(missing_ok=True)
    shutil.rmtree(shim, ignore_errors=True)

TABLE_NAMES = ("proton_scores", "enriching_mutations", "depleting_mutations",
               "stabilizing_enriching_mutations", "stabilizing_depleting_mutations",
               "pssm_enriching", "pssm_depleting")
RUN_LABEL = f"{PDB_NAME} chain {chain_ID}"
RUNS[RUN_LABEL] = {
    "result_dir": RESULT_DIR, "prefix": prefix, "algorithm": algorithm, "ddg": f"DDG_{algorithm}_Scores",
    "cut_off": cut_off, "IQR": IQR,
    "tables": {name: (pd.read_csv(RESULT_DIR / f"{prefix}_{name}", sep=" ")
                      if (RESULT_DIR / f"{prefix}_{name}").is_file() else None) for name in TABLE_NAMES},
}

minutes, seconds = divmod(int(time.time() - started), 60)
print(f"PROT-ON finished in {minutes} min {seconds} s.")
print("Results folder:", RESULT_DIR)
if len(RUNS) > 1:
    print(f"This session now holds {len(RUNS)} runs: {', '.join(RUNS)}. Step 5 compares them.")

## 4. Results

**Reading the tables.** `DDG_<algorithm>_Scores` is the binding score of the mutant minus that of the wild type: negative values are predicted to enrich binding, positive values to deplete it. `DDG_Stability_Scores` is the same difference for the stability of the scanned chain alone: negative values are stabilizing. The *designer mutations* are the enriching and depleting outliers that are also stabilizing.

In [ ]:
#@title Designer mutations { display-mode: "form" }
from IPython.display import Image, Markdown, display

if RESULT_DIR is None:
    raise RuntimeError("Run step 3 first.")
tables = RUNS[RUN_LABEL]["tables"]
scores = tables["proton_scores"]
ddg = RUNS[RUN_LABEL]["ddg"]

display(Markdown(
    f"**{len(scores)} mutations** were scanned at **{scores['Positions'].nunique()} interface positions** "
    f"of chain {chain_ID}. Binding-enriching outliers: **{len(tables['enriching_mutations'])}** "
    f"({len(tables['stabilizing_enriching_mutations'])} also stabilizing). "
    f"Binding-depleting outliers: **{len(tables['depleting_mutations'])}** "
    f"({len(tables['stabilizing_depleting_mutations'])} also stabilizing)."))

titles = {"stabilizing_enriching_mutations": "Designer mutations: binding enriching and stabilizing",
          "stabilizing_depleting_mutations": "Designer mutations: binding depleting and stabilizing",
          "pssm_enriching": "Enriching designer mutations that also pass the PSSM filter",
          "pssm_depleting": "Depleting designer mutations that also pass the PSSM filter"}
for name, title in titles.items():
    table = tables[name]
    if table is None:
        continue
    display(Markdown(f"#### {title}"))
    if table.empty:
        print("None found.")
    else:
        display(table.sort_values(ddg).reset_index(drop=True))

In [ ]:
#@title Figures { display-mode: "form" }
figures = {"heatmap": "Heatmap of the ΔΔG scores (outliers are saturated)",
           "boxplot": "Distribution of all ΔΔG scores",
           "PositionSpecificBoxPlot": "ΔΔG scores per position",
           "ResidueSpecificBoxPlot": "ΔΔG scores per mutant amino acid"}
for name, title in figures.items():
    path = RESULT_DIR / f"{prefix}_{name}.png"
    if path.is_file():
        display(Markdown(f"#### {title}"))
        display(Image(filename=str(path), width=700))

In [ ]:
#@title Interactive heatmap (hover to read the values) { display-mode: "form" }
import plotly.express as px

matrix = scores.pivot_table(index="Positions", columns="Mutations", values=ddg)
order = sorted(matrix.index, key=lambda position: int(re.search(r"-?\d+", position).group()))
figure = px.imshow(matrix.loc[order], color_continuous_scale="RdBu_r", color_continuous_midpoint=0,
                   aspect="auto", labels=dict(x="Mutation", y="Position", color=f"{algorithm} ΔΔG"))
figure.update_layout(width=750, height=max(400, 22 * len(order) + 150),
                     title=f"Chain {chain_ID} of {PDB_NAME}: all scanned mutations")
figure.show()

## 5. Compare the structures you have scanned

How much a predicted mutation depends on the structure it was read from is worth knowing: a docking model and an experimental structure of the same complex do not always propose the same mutations. This step puts every run of this session side by side. It compares them by residue number, so it is meaningful only for runs of the same complex, numbered the same way.

To fill it, run step 3 once per structure, changing `pdb_file` in step 2 in between, then run the cell below.

In [ ]:
#@title Compare the runs of this session { display-mode: "form" }
import numpy as np
import plotly.express as px

if len(RUNS) < 2:
    display(Markdown(
        "This session holds **one run**. To compare structures of the same complex — an experimental structure "
        "and docking models of it, for instance — go back to step 2, choose another structure, run step 3 again, "
        "and then run this cell."))
else:
    labels = list(RUNS)
    ddg_table = pd.DataFrame({
        label: pd.Series(run["tables"]["proton_scores"][run["ddg"]].to_numpy(),
                         index=run["tables"]["proton_scores"]["Positions"].astype(str)
                               + run["tables"]["proton_scores"]["Mutations"].astype(str))
        for label, run in RUNS.items()})

    def _designer(run, kind):
        table = run["tables"][f"stabilizing_{kind}_mutations"]
        if table is None or table.empty:
            return set()
        return set(table["Positions"].astype(str) + table["Mutations"].astype(str))

    for kind, wording in (("enriching", "enrich"), ("depleting", "deplete")):
        picked = {label: _designer(run, kind) for label, run in RUNS.items()}
        union = sorted(set().union(*picked.values()))
        display(Markdown(f"#### Designer mutations predicted to {wording} binding"))
        if not union:
            print("None in any of the runs.")
            continue
        summary = ddg_table.reindex(union).round(2)
        summary.insert(0, "predicted in", [", ".join(l for l in labels if mutation in picked[l])
                                           for mutation in union])
        summary.index.name = "Mutation"
        display(summary)
        shared = set.intersection(*picked.values())
        alone = sum(1 for mutation in union if sum(mutation in picked[l] for l in labels) == 1)
        print(f"Predicted in every run: {', '.join(sorted(shared)) if shared else 'none'}. "
              f"Predicted in a single run only: {alone} of {len(union)}.")

    common = ddg_table.dropna()
    display(Markdown(f"#### How far the runs agree, over the {len(common)} mutations scanned in all of them"))
    if len(common) > 2:
        display(pd.DataFrame([[round(float(np.corrcoef(common[a], common[b])[0, 1]), 2) for b in labels]
                              for a in labels], index=labels, columns=labels))
        reference = labels[0]
        long = pd.concat([pd.DataFrame({"reference": common[reference], "other": common[label],
                                        "run": label, "mutation": common.index})
                          for label in labels if label != reference])
        figure = px.scatter(long, x="reference", y="other", facet_col="run", hover_name="mutation",
                            template="plotly_white", opacity=0.6,
                            labels={"reference": f"ΔΔG in {reference}", "other": "ΔΔG in the other run"})
        for annotation in figure.layout.annotations:
            annotation.text = annotation.text.split("=")[-1]
        limits = [float(common.to_numpy().min()), float(common.to_numpy().max())]
        figure.add_shape(type="line", x0=limits[0], y0=limits[0], x1=limits[1], y1=limits[1],
                         line=dict(dash="dot", width=1), row="all", col="all")
        figure.update_layout(height=430, showlegend=False,
                             title="Each point is one mutation; the dotted line is perfect agreement")
        figure.show()
    else:
        print("The runs have too few mutations in common to compare. Do they scan the same chain of the same complex?")

## 6. Download the results
Packs the results folder into a zip file: the score tables, the lists of enriching and depleting mutations, the figures (png and svg), the run log and, optionally, the structure of every mutant. In Colab the download starts automatically.

In [ ]:
#@title Download the results { display-mode: "form" }
include_mutation_models = True #@param {type:"boolean"}
import zipfile

if RESULT_DIR is None:
    raise RuntimeError("Run step 3 first.")
run_folder = RESULT_DIR.parent
archive = run_folder.with_suffix(".zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULT_DIR.rglob("*")):
        if path.is_file() and (include_mutation_models or "mutation_models" not in path.parts):
            bundle.write(path, Path(run_folder.name) / path.relative_to(RESULT_DIR))
print(f"Saved {archive} ({archive.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))